# 33장 · LangChain 검색 도구와 ToolMessage

“대출 기간은?”이라는 질문을 검색 도구에 보냈습니다. 실행 기록에는 search-1과 library-1이 함께 나옵니다. 하나는 방금 보낸 요청을, 다른 하나는 대답의 근거를 가리킵니다. 이번에는 도구를 실행할 노드를 직접 연결한 뒤, 같은 자리에 ToolNode를 넣어 두 기록을 비교합니다.

## 실습 노트북과 실행 환경

Python 3 커널로 노트북을 열고 위에서부터 실행하세요. 기본 실습은 키 없이 진행합니다. 도구를 요청하는 메시지는 미리 만들어 두고, 문서 검색과 그래프 실행은 노트북에서 직접 합니다. 마지막 선택 실습에서 실제 모델을 연결할 수 있습니다.

실행 환경은 Python 3.12.15, langchain-core 1.6.12, langgraph 1.2.16, langgraph-prebuilt 1.1.1, pydantic 2.14.1입니다.

## 검색 함수에 도구 이름과 인자 붙이기

30장의 가상 도서관 문서 세 건과 검색 함수입니다. 준비 셀을 한 번 실행하세요. 대출·휴관·반납 중 질문에 들어 있는 낱말로 문서를 찾습니다. 이번 장에서 바꿀 곳은 이 함수를 호출하는 쪽입니다.

In [1]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]


### 셀 01 · 도구가 받을 인자 확인하기

모델에 어떤 검색을 요청할 수 있다고 알려 줄까요? 이름은 `search_documents`, 검색 문장은 `question`, 돌려줄 문서 수의 상한은 `k`로 정합니다. 아래 셀에서 이름과 필수 인자 목록을 확인하세요.

In [2]:
from langchain_core.tools import tool
from langchain_core.messages import AIMessage
from pydantic import BaseModel, ConfigDict, StrictInt, StrictStr, Field

class SearchArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    question: StrictStr
    k: StrictInt = Field(default=2, ge=1)

@tool("search_documents", args_schema=SearchArgs, response_format="content_and_artifact")
def search_documents(question: str, k: int = 2):
    """가상 도서관의 대출·휴관·반납 안내를 찾습니다."""
    if not question.strip():
        raise ValueError("질문을 입력하세요.")
    found = retrieve(question, k)
    return "\n".join(row["text"] for row in found), found

print(search_documents.name, SearchArgs.model_json_schema()["required"])

search_documents ['question']


목록에는 question만 나옵니다. k를 생략하면 기본값 2를 사용합니다. StrictInt와 ge=1은 각각 정수인지, 1 이상인지 검사합니다. k에 문자열 "1"을 넣으면 검색을 시작하기 전에 인자 검사에서 멈춥니다.

함수의 반환문도 살펴보세요. 첫째 값은 모델이 읽을 안내 문장이고, 둘째 값은 문서 ID와 쪽수가 남은 레코드 목록입니다. 둘째 값을 따로 보관하려고 response_format을 content_and_artifact로 지정했습니다.

### 셀 02 · 요청에 적힌 값 읽기

대출 기간을 묻는 모델 응답을 한 건 준비합니다. 아래에서 직접 정한 search-1은 이 요청의 이름표입니다. 뒤에서 두 노드를 비교할 때도 이 요청을 그대로 씁니다.

In [3]:
request = AIMessage(content="", tool_calls=[{
    "name": "search_documents",
    "args": {"question": "대출 기간은?", "k": 1},
    "id": "search-1",
    "type": "tool_call",
}])
call = request.tool_calls[0]
print(call["name"], call["id"], call["args"])

search_documents search-1 {'question': '대출 기간은?', 'k': 1}


출력에는 질문과 요청 ID가 보입니다. 여기서는 AIMessage만 만들었으므로 아직 검색된 문서는 없습니다. 처음 보는 용어라면 16장의 도구 요청과 실행 예제를 함께 보세요.

## 같은 검색을 두 방식으로 호출하기

### 셀 03 · 인자만 건넸을 때 남는 값

`invoke`에 질문과 k만 건넵니다. 대출 안내 문장을 얻은 뒤, 이 값의 자료형을 확인하세요.

In [4]:
text = search_documents.invoke(call["args"])
print(type(text).__name__, text)

str 도서 대출 기간은 14일입니다.


str은 일반 문자열입니다. 안내 문장은 읽을 수 있지만 문자열에서 tool_call_id나 artifact를 꺼낼 수는 없습니다. 이번에는 같은 검색을 호출 ID까지 포함해 실행해 보세요.

### 과제 04 · 요청 ID와 근거 문서의 쪽수 찾기

`call` 전체를 넘기고, 받은 메시지에서 요청 ID와 첫 문서의 ID·쪽수를 출력하세요. 어느 값이 질문할 때 이미 정해져 있었는지도 짚어 보세요.

In [5]:
# reply에 호출 결과를 담으세요.
# reply.tool_call_id와 첫 문서의 id, page를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>reply = search_documents.invoke(call)
doc = reply.artifact[0]
print(reply.tool_call_id, doc[&quot;id&quot;], doc[&quot;page&quot;])</code></pre>
<p>실행 결과</p><pre><code>search-1 library-1 1</code></pre>
<p>search-1은 셀 02에서 정한 호출 ID이고, library-1과 1은 검색이 찾아낸 문서의 ID와 쪽수입니다. 다른 요청이 같은 문서를 찾으면 두 호출의 ID는 달라도 근거 문서의 ID는 같습니다.</p>
</details>

#### 문장은 같은데 요청 ID는 어디에 있을까요?

<mark>호출 ID가 필요하면 invoke에 호출 전체를 건네세요.</mark> 도구가 받은 값에 search-1이 있어야 결과 메시지에도 같은 ID를 붙일 수 있습니다. 셀 03의 문자열과 과제 04의 ToolMessage를 나란히 놓고, 어느 쪽에서 문서의 쪽수까지 확인할 수 있는지 보세요.

## 검색 도구를 실행하는 노드 직접 만들기

### 셀 05 · 상태에서 요청을 꺼내 도구 실행하기

그래프에서 노드는 지금까지의 메시지를 `state["messages"]`로 받습니다. 마지막 메시지에서 호출을 하나씩 꺼내고, 호출에 적힌 이름으로 실행할 도구를 고릅니다. 각 호출의 name이 코드에서 어디에 쓰이는지 따라가 보세요.

In [6]:
from langgraph.graph import MessagesState

tools_by_name = {search_documents.name: search_documents}

def run_tools(state: MessagesState):
    replies = []
    for tool_call in state["messages"][-1].tool_calls:
        selected = tools_by_name[tool_call["name"]]
        replies.append(selected.invoke(tool_call))
    return {"messages": replies}

update = run_tools({"messages": [request]})
print([message.type for message in update["messages"]])

['tool']


tool은 새로 만든 결과 메시지의 타입입니다. run_tools가 돌려준 목록에는 검색 결과만 들어 있습니다. 이 함수를 그래프의 노드로 연결하면 MessagesState가 새 결과를 원래 요청 뒤에 붙입니다. 셀 07에서 두 메시지가 모두 남는지 확인하세요.

### 호출 유무에 따른 실행 경로

```mermaid
flowchart TD
    start["START · 마지막 메시지 읽기"] --> check{"tool_calls가 있는가?"}
    check -->|있음| tools["tools · 등록한 도구 실행"]
    check -->|없음| stop["END"]
    tools -->|결과 메시지 추가| stop
```

두 그래프가 함께 쓰는 경로입니다. 요청이 있으면 tools를 한 번 거쳐 종료하고, 요청이 없으면 바로 종료합니다. tools 자리에 직접 만든 함수와 ToolNode를 차례로 넣습니다.

### 셀 06 · 호출이 있을 때만 실행하는 그래프 연결하기

`executor`는 tools 자리에 넣을 함수나 노드입니다. 경로를 정하는 줄과 그 경로에서 실행할 코드를 등록하는 줄을 찾아보세요.

In [7]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def build_executor(executor):
    graph = StateGraph(MessagesState)
    graph.add_node("tools", executor)
    graph.add_conditional_edges(
        START, tools_condition, {"tools": "tools", "__end__": END}
    )
    graph.add_edge("tools", END)
    return graph.compile()

manual_graph = build_executor(run_tools)
print(tools_condition({"messages": [request]}))

tools


tools가 찍힌 이유는 마지막 AIMessage에 요청이 한 건 있기 때문입니다. 조건 함수가 돌려준 "tools"는 간선 표의 왼쪽 키이고, 오른쪽 "tools"는 다음 노드 이름입니다. "__end__"가 나오면 END로 갑니다. compile은 연결을 확정합니다. 실제 검색은 다음 셀의 invoke에서 시작합니다.

### 셀 07 · ToolNode로 바꾸고 같은 요청 다시 보내기

두 그래프에 똑같은 request를 넣습니다. 입력부터 마지막 결과까지 메시지 타입을 읽고, 어느 요청이 어느 문서를 찾았는지 비교하세요.

In [8]:
standard_graph = build_executor(ToolNode([search_documents]))

for label, graph in [("직접 만든 노드", manual_graph), ("ToolNode", standard_graph)]:
    history = graph.invoke({"messages": [request]})["messages"]
    reply = history[-1]
    print(label, [m.type for m in history], reply.tool_call_id, reply.artifact[0]["id"])

직접 만든 노드 ['ai', 'tool'] search-1 library-1
ToolNode ['ai', 'tool'] search-1 library-1


두 줄 모두 ai 뒤에 tool이 붙고 search-1, library-1이 나옵니다. 셀 05에서 직접 썼던 이름 조회, 호출 순회, 결과 메시지 반환을 이번에는 ToolNode가 맡았습니다. 입력 질문, 검색 함수, 조건부 간선과 종료 위치는 두 실행에서 같습니다.

#### ToolNode가 있어도 검색을 건너뛸 수 있을까요?

<mark>도구를 등록하는 줄과 검색 여부를 가르는 줄은 서로 다릅니다.</mark> 셀 07의 ToolNode에 검색 함수를 등록했습니다. 검색을 생략할 수 있는 이유는 셀 06의 조건부 간선이 요청 없는 메시지를 END로 보내기 때문입니다. 과제 09에서 두 경로를 각각 실행해 보세요.

### 과제 08 · 쪽수로 찾는 도구를 같은 그래프에 연결하기

이용자가 “안내문 3쪽을 읽어 주세요”라고 부탁했습니다. 이번에는 검색어를 고를 필요 없이 쪽수를 알면 됩니다. 제공된 도구 read_page를 search_documents와 함께 등록하고 새 그래프를 만드세요. page-3 요청을 보내서 등록된 도구 중 어느 도구가 응답했는지 이름, 호출 ID, 안내 문장으로 확인하세요.

In [9]:
@tool
def read_page(page: int) -> str:
    """가상 도서관 안내문에서 지정한 쪽의 문장을 읽습니다."""
    for row in records:
        if row["page"] == page:
            return row["text"]
    return "해당 쪽이 없습니다."

page_request = AIMessage(content="", tool_calls=[{
    "id": "page-3", "name": "read_page", "args": {"page": 3}, "type": "tool_call",
}])
# 두 도구를 받는 ToolNode를 만들고 build_executor로 그래프에 연결하세요.
# page_request를 실행한 뒤 마지막 메시지의 name, tool_call_id, content를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>@tool
def read_page(page: int) -&gt; str:
    &quot;&quot;&quot;가상 도서관 안내문에서 지정한 쪽의 문장을 읽습니다.&quot;&quot;&quot;
    for row in records:
        if row[&quot;page&quot;] == page:
            return row[&quot;text&quot;]
    return &quot;해당 쪽이 없습니다.&quot;

page_request = AIMessage(content=&quot;&quot;, tool_calls=[{
    &quot;id&quot;: &quot;page-3&quot;, &quot;name&quot;: &quot;read_page&quot;, &quot;args&quot;: {&quot;page&quot;: 3}, &quot;type&quot;: &quot;tool_call&quot;,
}])
two_tools = ToolNode([search_documents, read_page])
page_graph = build_executor(two_tools)
reply = page_graph.invoke({&quot;messages&quot;: [page_request]})[&quot;messages&quot;][-1]
print(reply.name, reply.tool_call_id, reply.content)</code></pre>
<p>실행 결과</p><pre><code>read_page page-3 무인 반납함은 24시간 이용할 수 있습니다.</code></pre>
<p>read_page가 찍힌 까닭은 요청의 name이 read_page였기 때문입니다. ToolNode가 그 이름에 맞는 도구를 골라 page=3을 건넸습니다. 이 도구는 문장 하나를 반환하므로 content에서 반납함 안내를 바로 읽을 수 있습니다. 같은 그래프에 셀 02의 request를 보내면 search_documents도 여전히 사용할 수 있습니다.</p>
</details>

## 실행 기록으로 세 가지 상황 구별하기

### 과제 09 · 검색 생략과 검색 결과 없음 구분하기

인사에는 호출을 넣지 않고, 주차 요금 질문에는 search-parking이라는 호출을 넣습니다. 두 입력을 각각 standard_graph에 보낸 뒤, tool 타입 메시지의 개수와 각 메시지의 artifact를 출력하세요. 주차 자료가 없을 때에도 검색 노드를 지나는지 확인하세요.

In [10]:
greeting = AIMessage(content="안녕하세요!")
parking_call = {**call, "id": "search-parking", "args": {"question": "주차 요금은?", "k": 1}}
parking = AIMessage(content="", tool_calls=[parking_call])
# 두 입력을 실행하고 tool 메시지 개수와 artifact 목록을 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>greeting = AIMessage(content=&quot;안녕하세요!&quot;)
parking_call = {**call, &quot;id&quot;: &quot;search-parking&quot;, &quot;args&quot;: {&quot;question&quot;: &quot;주차 요금은?&quot;, &quot;k&quot;: 1}}
parking = AIMessage(content=&quot;&quot;, tool_calls=[parking_call])
for label, message in [(&quot;인사&quot;, greeting), (&quot;주차&quot;, parking)]:
    history = standard_graph.invoke({&quot;messages&quot;: [message]})[&quot;messages&quot;]
    results = [m for m in history if m.type == &quot;tool&quot;]
    print(label, len(results), [m.artifact for m in results])</code></pre>
<p>실행 결과</p><pre><code>인사 0 []
주차 1 [[]]</code></pre>
<p>인사의 0 []는 tool 메시지가 한 건도 없다는 뜻입니다. 주차 질문의 1 [[]]에서 바깥 목록은 도구 응답 한 건, 안쪽 빈 목록은 찾은 문서가 0건이라는 뜻입니다. 여기서 artifact[0]을 읽으면 IndexError가 납니다. 학생 화면에 ‘안내 자료가 없습니다’를 띄울 자리는 두 번째 경로입니다.</p>
</details>

### 과제 10 · 두 요청을 각각의 결과에 연결하기

한 번의 모델 응답에 요청이 두 건 들어올 수도 있습니다. 아래 두 호출을 한 AIMessage에 담아 standard_graph를 한 번 실행하세요. 결과 메시지에서 호출 ID를 키로 삼아 문서 ID를 찾고 휴관, 대출 순서로 출력하세요.

In [11]:
closed_call = {**call, "id": "search-2", "args": {"question": "휴관일은?", "k": 1}}
# call과 closed_call을 한 메시지로 보내세요.
# 결과를 요청 ID로 찾아 search-2, search-1 순서로 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>closed_call = {**call, &quot;id&quot;: &quot;search-2&quot;, &quot;args&quot;: {&quot;question&quot;: &quot;휴관일은?&quot;, &quot;k&quot;: 1}}
batch = AIMessage(content=&quot;&quot;, tool_calls=[call, closed_call])
history = standard_graph.invoke({&quot;messages&quot;: [batch]})[&quot;messages&quot;]
by_request = {m.tool_call_id: m.artifact for m in history if m.type == &quot;tool&quot;}
for request_id in [&quot;search-2&quot;, &quot;search-1&quot;]:
    print(request_id, by_request[request_id][0][&quot;id&quot;])</code></pre>
<p>실행 결과</p><pre><code>search-2 library-2
search-1 library-1</code></pre>
<p>두 번째로 보낸 요청을 먼저 조회해도 library-2가 나옵니다. 검색 함수 이름은 두 요청 모두 search_documents라서 이름만으로는 어느 질문의 답인지 구별할 수 없습니다. 각 결과가 돌려준 tool_call_id로 묶으면 출력 순서를 바꿔도 짝이 맞습니다.</p>
</details>

### 셀 11 · 실행할 도구를 찾을 수 없는 경우

이번에는 자료와 질문을 그대로 두고 도구 이름에 오타 하나를 넣습니다. 두 그래프에서 문제가 어디에 나타날지 먼저 예상하세요.

In [12]:
typo = AIMessage(content="", tool_calls=[{**call, "name": "search_document"}])
try:
    manual_graph.invoke({"messages": [typo]})
except KeyError as exc:
    print("직접 만든 노드", type(exc).__name__, exc.args[0])

failed = standard_graph.invoke({"messages": [typo]})["messages"][-1]
print("ToolNode", failed.type, failed.status, failed.tool_call_id)

직접 만든 노드 KeyError search_document
ToolNode tool error search-1


직접 만든 노드는 등록 표에서 이름을 찾다가 KeyError로 멈춥니다. 이 버전의 ToolNode는 등록 목록에서 이름을 찾지 못하면 status가 error인 ToolMessage를 돌려줍니다. search-1이 있어도 검색이 성공했다는 뜻은 아닙니다. 여기서는 요청에 쓴 도구 이름을 고치세요. 과제 09의 주차 질문에서는 검색이 끝났으므로 안내할 문서가 있는지 살펴보세요.

## 선택 실습 · 실제 모델에서 검색 요청 받기

여기까지는 여러분이 요청을 직접 만들었습니다. 실제 모델에 연결할 때는 모델에도 도구 이름과 인자 형식을 알려 줍니다. 다음 함수는 사용자의 질문을 받고, 모델이 돌려준 첫 응답을 방금 만든 그래프에 그대로 넣습니다. 검색을 했다면 모델이 결과를 읽고 이용자에게 전할 문장을 한 번 작성합니다.

### 셀 12 · 모델에 명세를 건네고 검색 결과 돌려주기

이 셀은 수업에서 쓸 함수를 정의합니다. model에는 도구 호출을 지원하는 채팅 모델을 넣습니다. `bind_tools`와 `ToolNode`에 같은 도구를 건네는 두 곳을 찾아보세요.

In [13]:
from langchain_core.messages import HumanMessage

def ask_library(question, model):
    user = HumanMessage(content=question)
    model_with_tools = model.bind_tools([search_documents])
    decision = model_with_tools.invoke([user])
    history = standard_graph.invoke({"messages": [user, decision]})["messages"]
    if not decision.tool_calls:
        return decision, history
    answer = model.invoke(history)
    return answer, history

bind_tools로 모델이 검색 도구의 명세를 읽을 수 있게 합니다. 검색을 요청할지는 질문을 읽은 모델이 정합니다. 모델의 응답을 받은 그래프는 호출이 있을 때 ToolNode를 실행합니다. 호출이 없으면 첫 응답을 그대로 돌려줍니다. 호출이 있으면 원래 질문, 호출과 도구 결과를 함께 전달해 최종 답변을 받습니다.

### 모델에 도구 결과를 돌려주는 경로

```mermaid
flowchart TD
    ask["사용자 질문"] --> model["도구 명세를 받은 모델"]
    model --> route_input["질문과 첫 응답을 그래프에 전달"]
    route_input --> choice{"첫 응답에 도구 호출이 있는가?"}
    choice -->|없음| final["첫 응답 반환"]
    choice -->|있음| tools["그래프에서 도구 실행"]
    tools --> compose["결과를 읽은 모델이 답변 작성"]
```

두 경우 모두 질문과 첫 응답을 그래프에 보냅니다. 호출이 없으면 그래프는 바로 끝나고, history의 마지막 메시지가 답변입니다. 호출이 있으면 history의 마지막 메시지는 도구 결과입니다. 함수는 이 기록으로 모델을 한 번 더 호출하고, 새로 작성한 답변을 answer로 돌려줍니다.

### 셀 13 · 모델이 검색을 선택했는지 확인하기

수업에서 설정한 채팅 모델 객체가 있으면 class_model에 넣으세요. OpenAI 수업 계정이라면 커널에 langchain-openai를 설치하고 OPENAI_API_KEY를 설정한 뒤, 아래 두 줄의 주석을 풀고 수업에서 지정한 모델명을 넣으세요. 이 선택 실습은 해당 모델을 실제로 호출합니다. 인사와 대출 질문을 각각 넣고, 답변 문장과 함께 기록에 tool이 남았는지 확인하세요.

In [14]:
class_model = None
# from langchain_openai import ChatOpenAI
# class_model = ChatOpenAI(model="수업에서 지정한 모델명")
if class_model is None:
    print("모델을 연결하면 이 셀에서 질문을 보냅니다.")
else:
    for question in ["안녕하세요!", "도서관 안내를 찾아 대출 기간을 알려 주세요."]:
        answer, history = ask_library(question, class_model)
        print(question, [m.type for m in history])
        print(answer.content)

모델을 연결하면 이 셀에서 질문을 보냅니다.


기본 설정에서 보이는 한 줄은 모델을 연결하라는 안내입니다. 실제 연결 뒤에는 모델의 선택에 따라 기록이 달라집니다. 대출 기간이 답에 나와도 검색을 했다고 단정할 수 없습니다. history에 tool이 있으면 그 메시지의 tool_call_id와 artifact에서 요청과 근거를 확인하세요.

## 혼자 설명해 보기

동료가 “문서가 0개면 모델이 검색을 건너뛴 거죠?”라고 묻습니다. 과제 09의 주차 질문 실행 기록에서 근거를 찾아 답해 보세요. 이어서 과제 10의 두 질문이 같은 도구를 썼는데도 결과를 구별할 수 있었던 이유를 코드 한 줄로 가리켜 보세요.

오늘 만든 그래프는 첫 모델 응답의 요청을 처리하고 끝납니다. 34장에서는 검색 결과를 받은 뒤 다시 판단하는 경로를 그래프 안에 연결하고 검색 횟수를 비교합니다.

## 참고 자료

[도구 이름·입력 스키마와 도구 호출](https://docs.langchain.com/oss/python/langchain/tools)

[AIMessage와 ToolMessage](https://docs.langchain.com/oss/python/langchain/messages)

[LangGraph의 도구 노드와 조건부 간선](https://docs.langchain.com/oss/python/langgraph/quickstart)

[선택 실습의 ChatOpenAI 설정](https://docs.langchain.com/oss/python/integrations/chat/openai)